# Network ReAct Agent - Demo Notebook
"Reasoning Loop with External State"

## Setup

In [ ]:
import sys
from pathlib import Path

# Add src to path so we can import the package
sys.path.insert(0, str(Path(__file__).parent.parent / "src"))

from network_management_agent import (
    LLMConfig,
    create_llm,
    load_data,
    build_agent,
)

## Load Data

In [ ]:
candidates, members = load_data()
print(f"Loaded {len(candidates)} candidates, {len(members)} members")

## Create LLM

In [ ]:
llm_config = LLMConfig()
llm = create_llm(llm_config)

# Quick test
response = llm.invoke("Hey there! Just making sure you're up and running!")
response.pretty_print()

## Build Agent

In [ ]:
agent = build_agent(llm, candidates, members)
print("Agent built successfully!")

## Visualize Graph

In [ ]:
from IPython.display import Image, display
display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

## Run the Agent

In [ ]:
import json
from datetime import datetime
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage


def run_agent_demo(agent, inputs: dict, config: dict):
    """Run the agent and stream output to console."""
    print("Starting Agent Execution...\n")

    for chunk in agent.stream(inputs, stream_mode="updates", config=config):
        for node_name, output in chunk.items():
            print(f"\n>>>> NODE: {node_name} <<<<")
            if node_name in ("network_manager", "tools"):
                if "messages" in output:
                    for m in output["messages"]:
                        print("   --- CURRENT CONTENT ---")
                        m.pretty_print()

                        if hasattr(m, 'tool_calls') and m.tool_calls:
                            print("   --- TOOL CALL ---")
                            for tc in m.tool_calls:
                                print(f"   -> {tc['name']}({json.dumps(tc['args'], default=str)})")

    # Save action log
    history = list(agent.get_state_history(config))
    history.reverse()

    last_message_id = None
    filename = f"react_agent_actions_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

    with open(filename, "w", encoding="utf-8") as f:
        f.write("=== AGENT ACTION LOG ===\n\n")

        for i, state in enumerate(history):
            messages = state.values.get("messages", [])
            if not messages:
                continue

            m = messages[-1]

            if getattr(m, "id", None) == last_message_id:
                node_type = state.metadata.get("node", "update_state")
                if node_type in ("update_state", "summarize_messages"):
                    continue
                f.write(f"[STEP {i}] STATE UPDATE\n")
                f.write(f"  Node executed: {node_type}\n\n")
                continue

            last_message_id = getattr(m, "id", None)

            if isinstance(m, AIMessage) and m.tool_calls:
                f.write(f"[STEP {i}] AI -> TOOL CALL\n")
                for tc in m.tool_calls:
                    f.write(f"  -> {tc['name']}({json.dumps(tc['args'], default=str)})\n")
                f.write("\n")
                continue

            if isinstance(m, ToolMessage):
                content = m.content
                if isinstance(content, list):
                    content = " ".join(str(c) for c in content)
                f.write(f"[STEP {i}] TOOL RESULT ({m.name})\n")
                f.write(f"  {content.strip()}\n\n")
                continue

            if isinstance(m, AIMessage):
                content = m.content
                if isinstance(content, list):
                    content = " ".join(str(c) for c in content)
                if content.strip():
                    f.write(f"[STEP {i}] AI OUTPUT\n")
                    f.write(f"  {content.strip()}\n\n")
                continue

            if isinstance(m, HumanMessage):
                content = m.content
                if isinstance(content, list):
                    content = " ".join(str(c) for c in content)
                f.write(f"[STEP {i}] HUMAN INPUT\n")
                f.write(f"  {content.strip()}\n\n")
                continue

    print(f"\nExecution Complete. Detailed history saved to '{filename}'")

In [ ]:
# Create a thread
config = {"configurable": {"thread_id": "1"}}

# Specify an input
messages = [HumanMessage(content='''
Please add one high effectiveness hospital to the network.
''')]

inputs = {
    "messages": messages,
    "candidates": candidates,
    "members": members
}

run_agent_demo(agent, inputs, config)

## Inspect Agent State

In [ ]:
print("Original message:")
print(agent.get_state(config).values.get("original_message", ""))

In [ ]:
print("Summary:")
print(agent.get_state(config).values.get("summary", ""))

In [ ]:
import pandas as pd
print("Current network:")
pd.DataFrame(agent.get_state(config).values.get("network", []))

## Try a More Complex Prompt

In [ ]:
# Reset with a new thread
config = {"configurable": {"thread_id": "2"}}

messages = [HumanMessage(content='''
OK, now I'd like to add hospitals so that each cluster has a hospital. 
Please add high effectiveness hospitals for each cluster. 
I know you can't see all the clusters directly, but if you call get_candidates 
a few times, you'll eventually see them. Then report back when done, with the status of the network.
''')]

inputs = {
    "messages": messages,
    "candidates": candidates,
    "members": members
}

run_agent_demo(agent, inputs, config)

In [ ]:
# Check final state
print("Summary:")
print(agent.get_state(config).values.get("summary", ""))

print("\nNetwork:")
pd.DataFrame(agent.get_state(config).values.get("network", []))